New mosaics (and new bands), and updated data version numbers

In [3]:
import ee
import gdown
import pandas as pd

In [4]:
ee.Initialize(project='mapbiomas-india')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


1. Get total samples per year for each region

In [ ]:
regions = ['HIM', 'IGP', 'DES', 'PEN', 'NEI', 'WGR', 'ICR']
region_id_map = {'HIM': 1, 'IGP': 2, 'DES': 3, 'PEN': 4, 'NEI': 5, 'WGR': 6, 'ICR': 7}

# Define regional sample versions 
# (Update these values with your actual version strings)
samples_version_map = {
    'HIM': '7', 
    # # 'IGP': '7', 
    # # 'DES': '8', 
    # 'PEN': '6', 
    # 'NEI': '7', 
    # # 'WGR': '5', 
    # # 'ICR': '7'
}

In [ ]:
def get_region_sample_counts(
    year: int,
    samples_folder: str = 'projects/mapbiomas-india/assets/LAND-COVER/COLLECTION-1/SAMPLES/',
    region_id_map: dict = None,
    samples_version_map: dict = None,
    territory_name:str='INDIA'
) -> pd.DataFrame:
    """
    Computes total sample counts per region for a specified year
    and returns a formatted DataFrame.
    """
    if region_id_map is None:
        region_id_map = {'HIM': 1, 'IGP': 2, 'DES': 3, 'PEN': 4, 'NEI': 5, 'WGR': 6, 'ICR': 7}
    
    if samples_version_map is None:
        samples_version_map = {'HIM': '7', 'IGP': '7', 'DES': '8', 'WGR': '5', 'ICR': '7',  'PEN': '6', 'NEI': '7'}

    records = []

    for region, version in samples_version_map.items():
        region_id = region_id_map.get(region)
        if region_id is None:
            continue

        asset_name = f"{territory_name}_training_samples_region{region_id}_v{version}_merged"
        asset_path = f"{samples_folder.rstrip('/')}/{asset_name}"

        try:
            fc = ee.FeatureCollection(asset_path).filter(ee.Filter.eq('year', year))
            count = fc.size().getInfo()
        except ee.EEException as e:
            count = None

        records.append({
            'Region': region,
            'Region ID': region_id,
            'Version': version,
            'Year': year,
            'Total Samples': count
        })

    df = pd.DataFrame(records)
    return df

In [7]:
get_region_sample_counts(
    year = 2024,
    region_id_map = region_id_map,
    samples_version_map = samples_version_map
)

,Region,Region ID,Version,Year,Total Samples
0,HIM,1,7,2024,48920
1,PEN,4,6,2024,58260
2,NEI,5,7,2024,162288


In [8]:
get_region_sample_counts(
    year = 2015,
    region_id_map = region_id_map,
    samples_version_map = samples_version_map
)

,Region,Region ID,Version,Year,Total Samples
0,HIM,1,7,2015,48657
1,PEN,4,6,2015,58316
2,NEI,5,7,2015,133443


In [18]:
def export_samples_to_drive(
    regions: list,
    years: list = None,
    drive_folder: str = "GEE_EXPORTS",
    samples_folder: str = "projects/mapbiomas-india/assets/LAND-COVER/COLLECTION-1/SAMPLES/",
    region_id_map: dict = None,
    sample_version_map: dict = None,
    territory_name: str = 'INDIA'
):
    """
    Exports training samples to Drive.
    If `years` is None or empty, exports the complete un-filtered dataset (all years).
    """
    if region_id_map is None:
        region_id_map = {'HIM': 1, 'IGP': 2, 'DES': 3, 'PEN': 4, 'NEI': 5, 'WGR': 6, 'ICR': 7}
    if sample_version_map is None:
        sample_version_map = {'HIM': '7', 'IGP': '7', 'DES': '8', 'WGR': '5', 'ICR': '7', 'PEN': '6', 'NEI': '7'}

    for region in regions:
        if region not in sample_version_map:
            continue
        region_id = region_id_map[region]
        version = sample_version_map[region]
        asset_path = f"{samples_folder.rstrip('/')}/{territory_name}_training_samples_region{region_id}_v{version}_merged"

        # Case 1: Export complete asset across all years
        if not years:
            fc = (ee.FeatureCollection(asset_path)
                  .select(propertySelectors=['.*'], retainGeometry=False))
            description = f"{region}_all_years_v{version}"
            task = ee.batch.Export.table.toDrive(
                collection=fc,
                description=description,
                folder=drive_folder,
                fileNamePrefix=description,
                fileFormat='CSV'
            )
            task.start()
            print(f"Complete asset export task queued: {description} (Task ID: {task.id})")
            continue

        # Case 2: Export specific filtered years
        for year in years:
            fc = (ee.FeatureCollection(asset_path)
                  .filter(ee.Filter.eq('year', year))
                  .select(propertySelectors=['.*'], retainGeometry=False))

            description = f"{region}_{year}_v{version}"
            task = ee.batch.Export.table.toDrive(
                collection=fc,
                description=description,
                folder=drive_folder,
                fileNamePrefix=description,
                fileFormat='CSV'
            )
            task.start()
            print(f"Yearly export task queued: {description} (Task ID: {task.id})")

In [ ]:
# target_regions = ['DES']
# target_years = [1995,2005]

# export_samples_to_drive(
#     regions=target_regions,
#     years=target_years,
#     drive_folder="GEE_EXPORTS/IOLN_new_mosaics",
# )

Export task queued: DES_1995_v8 (Task ID: ZKAMHFXBHNWB7YCWEJY7T45M)
Export task queued: DES_2005_v8 (Task ID: FOOKDDY6VZQM3G53XQN7ESBV)


3. Download all years data for regions

In [19]:
target_regions = ['NEI','PEN']
target_years = None

In [20]:
export_samples_to_drive(
    regions=target_regions,
    years=None,
    drive_folder="GEE_EXPORTS/IOLN_new_mosaics_complete",
)

Complete asset export task queued: NEI_all_years_v7 (Task ID: H2ZVHRQ54FYG2GLNQIFPVXVI)
Complete asset export task queued: PEN_all_years_v6 (Task ID: NCXE3LNE2DRZ3E22WOO4YKCZ)


In [ ]:
target_regions = ['PEN']
target_years = None

export_samples_to_drive(
    regions=target_regions,
    years=None,
    drive_folder="GEE_EXPORTS/IOLN_new_mosaics_complete",
)


Download from drive to local

In [ ]:
# # Replace with your actual Google Drive folder ID
# folder_id = '1eM0jk5BsoaESATb_f9fsBT7N4mhj2ykF'

# # The function automatically handles nested folders and large files
# gdown.download_folder(id=folder_id, quiet=False, use_cookies=False)

In [21]:
# file_id = "1pbHqVi35Rjt4s0opctYlrrshNAugjABF"
# gdown.download(id=file_id)

Downloading...
From (original): https://drive.google.com/uc?id=1pbHqVi35Rjt4s0opctYlrrshNAugjABF
From (redirected): https://drive.google.com/uc?id=1pbHqVi35Rjt4s0opctYlrrshNAugjABF&confirm=t&uuid=3d6cdaae-fe40-48d7-80e5-3ee467ff6046
To: /home/nirzaree/Code/GitClones/IOLN/final_feature_selection/HIM_all_years_v7.csv
100%|██████████| 974M/974M [03:39<00:00, 4.44MB/s] 


'HIM_all_years_v7.csv'

In [ ]:
# file_id = "1PVB-upskTw88704wcIDr47I6whExNNN2"
# gdown.download(id=file_id)

Downloading...
From (original): https://drive.google.com/uc?id=1PVB-upskTw88704wcIDr47I6whExNNN2
From (redirected): https://drive.google.com/uc?id=1PVB-upskTw88704wcIDr47I6whExNNN2&confirm=t&uuid=bd021331-3df1-46dc-9c4d-8d884f6aa51f
To: /home/nirzaree/Code/GitClones/IOLN/final_feature_selection/PEN_all_years_v6.csv
100%|██████████| 1.24G/1.24G [06:29<00:00, 3.18MB/s]


'PEN_all_years_v6.csv'

In [25]:
# file_id = '1VWg5aoL3-Jy0DzEfuAXJ-h64JZQFRBIr'
# gdown.download(id=file_id)

Downloading...
From (original): https://drive.google.com/uc?id=1VWg5aoL3-Jy0DzEfuAXJ-h64JZQFRBIr
From (redirected): https://drive.google.com/uc?id=1VWg5aoL3-Jy0DzEfuAXJ-h64JZQFRBIr&confirm=t&uuid=a27abc46-0d8f-4ba9-bbea-a4e50e0565b5
To: /home/nirzaree/Code/GitClones/IOLN/final_feature_selection/NEI_all_years_v7.csv
100%|██████████| 2.24G/2.24G [10:50<00:00, 3.44MB/s] 


'NEI_all_years_v7.csv'

Convert All-Years Regional CSV or Parquet to Year-Partitioned Parquet
Partitions each regional dataset by `year` into compact, zstd-compressed Parquet files.

In [23]:
from pathlib import Path
import polars as pl
import re

In [26]:
input_folder = Path('data/regionwise_yearwise_samples/aggregated_regionwise_data/')
output_base_dir = Path('data/regionwise_yearwise_samples/')
output_base_dir.mkdir(parents=True, exist_ok=True)

# Discover both Parquet and CSV files
supported_extensions = ("*.parquet", "*.csv")
data_files = []
for ext in supported_extensions:
    data_files.extend(input_folder.glob(ext))
data_files = sorted(data_files)

print(f"Discovered {len(data_files)} source file(s): {[f.name for f in data_files]}")

for filepath in data_files:
    # Avoid processing a file if it was deleted as a counterpart in a previous iteration
    if not filepath.exists():
        continue

    # Extract region code (e.g., 'DES' from 'DES_all_years_v8.parquet' or 'DES_combined.csv')
    match = re.match(r"^([A-Za-z0-9]+)", filepath.stem)
    if not match:
        print(f"Skipping file with unrecognized naming format: {filepath.name}")
        continue

    region = match.group(1)
    region_partition_dir = output_base_dir / region

    # Skip if partitioned data already exists for this region
    if region_partition_dir.exists() and any(region_partition_dir.glob("year=*")):
        print(f"Partitioned data for '{region}' already exists at {region_partition_dir}. Skipping.")
        continue

    print(f"\nProcessing {filepath.name} for region '{region}'...")

    # Load data via lazy scan
    if filepath.suffix.lower() == ".parquet":
        lf = pl.scan_parquet(filepath)
    elif filepath.suffix.lower() == ".csv":
        lf = pl.scan_csv(filepath, infer_schema_length=10000)
    else:
        continue

    # Cast year to int32 to ensure clean directory names (year=1995/ vs year=1995.0/)
    schema = lf.collect_schema()
    if "year" in schema:
        lf = lf.with_columns(pl.col("year").cast(pl.Int32))
    else:
        raise KeyError(f"'year' column missing in {filepath.name}")

    df = lf.collect()
    raw_row_count = df.height
    years_present = sorted(df["year"].unique().to_list())
    print(f"  Found {raw_row_count:,} rows across {len(years_present)} years.")

    # Write Hive-partitioned Parquet files: {output_base_dir}/{region}/year=YYYY/
    region_partition_dir.mkdir(parents=True, exist_ok=True)
    df.write_parquet(
        region_partition_dir,
        partition_by=["year"],
        compression="zstd"
    )
    print(f"  Successfully partitioned '{region}' into: {region_partition_dir}/")

    # Integrity verification
    partitioned_row_count = (
        pl.scan_parquet(region_partition_dir / "**/*.parquet")
        .select(pl.len())
        .collect()
        .item()
    )

    if partitioned_row_count == raw_row_count:
        print(f"  Verification passed ({partitioned_row_count:,} rows match).")
        
        # 1. Delete the primary processed source file
        print(f"  Deleting processed source file: {filepath.name}...")
        filepath.unlink()

        # 2. Check and delete counterpart extension if present (e.g., .csv alongside .parquet)
        counterpart_ext = '.csv' if filepath.suffix.lower() == '.parquet' else '.parquet'
        counterpart_file = filepath.with_suffix(counterpart_ext)
        if counterpart_file.exists():
            print(f"  Deleting corresponding counterpart: {counterpart_file.name}...")
            counterpart_file.unlink()

        print(f"  Freed disk space successfully for '{region}'.")
    else:
        raise ValueError(
            f"Row count mismatch for {region}! Source: {raw_row_count:,}, "
            f"Partitioned: {partitioned_row_count:,}. Source files retained for debugging."
        )

Discovered 4 source file(s): ['DES_combined_multiyear.parquet', 'HIM_all_years_v7.csv', 'NEI_all_years_v7.csv', 'PEN_all_years_v6.csv']
Partitioned data for 'DES' already exists at data/regionwise_yearwise_samples/DES. Skipping.

Processing HIM_all_years_v7.csv for region 'HIM'...
  Found 1,714,482 rows across 39 years.
  Successfully partitioned 'HIM' into: data/regionwise_yearwise_samples/HIM/
  Verification passed (1,714,482 rows match).
  Deleting processed source file: HIM_all_years_v7.csv...
  Freed disk space successfully for 'HIM'.

Processing NEI_all_years_v7.csv for region 'NEI'...
  Found 3,851,207 rows across 39 years.
  Successfully partitioned 'NEI' into: data/regionwise_yearwise_samples/NEI/
  Verification passed (3,851,207 rows match).
  Deleting processed source file: NEI_all_years_v7.csv...
  Freed disk space successfully for 'NEI'.

Processing PEN_all_years_v6.csv for region 'PEN'...
  Found 2,158,251 rows across 39 years.
  Successfully partitioned 'PEN' into: data/